In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf

# ------------ Load the data from raw imports ------------ 
crsp = pd.read_csv('data/wrds_data.csv') 
hmi = pd.read_excel('data/hmi.xls', header=None) 
treasury = pd.read_csv('data/10_year_treasury.csv') 
compustat = pd.read_csv('data/compustat.csv')


# ------------  Data Cleaning and Aggregation ------------ 
# HMI: reshaping
data_start_idx = hmi[hmi[0] == 1985].index[0]
hmi = hmi.iloc[data_start_idx:].reset_index(drop=True)
hmi.columns = ['Year', 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
hmi['Year'] = hmi['Year'].astype(int)
hmi = hmi.melt(id_vars=['Year'], var_name='Month', value_name='HMI') # row per year -> row per month per year
hmi = hmi.sort_values(['Year', 'Month']).reset_index(drop=True)
hmi['HMI_lag1'] = hmi['HMI'].shift(1)
hmi = hmi.dropna(subset=['HMI_lag1']).reset_index(drop=True)

# FRED data; aggregating dailyt to monthly
treasury['date'] = pd.to_datetime(treasury['observation_date'], format='%Y-%m-%d')
treasury['Month'] = treasury['date'].dt.month
treasury['Year'] = treasury['date'].dt.year
treasury = treasury.groupby(['Year', 'Month'])['DGS10'].mean().reset_index() # from daily to monthly

# Compustat data; calcualting ratios
compustat['debt_to_assets'] = compustat['lt'] / compustat['at'] # Leverage: Total Liabilities / Total Assets
compustat['asset_growth'] = compustat.groupby('tic')['at'].pct_change() # Asset Growth: (Total Assets_t - Total Assets_{t-1}) / Total Assets_{t-1} 
compustat['log_assets'] = np.log(compustat['at']) # Size: log(Total Assets)
compustat['datadate'] = pd.to_datetime(compustat['datadate'], format='%Y-%m-%d')
compustat['Year'] = compustat['datadate'].dt.year
compustat = compustat[['tic', 'Year', 'debt_to_assets', 'asset_growth', 'log_assets']]

# REIT Data

#// add the HCN slicing thing; i don't think they are fully time consistent though like both exist at same time
# Does that mean that WELL ticker belonged to someone else prior to 2018?

crsp['date'] = pd.to_datetime(crsp['date'])
crsp['Month'] = crsp['date'].dt.month
crsp['Year'] = crsp['date'].dt.year
crsp['RET'] = pd.to_numeric(crsp['RET'], errors='coerce')
crsp['PRC'] = pd.to_numeric(crsp['PRC'], errors='coerce')
crsp = crsp.dropna(subset=['RET', 'PRC']).copy()
# Amihud (2002): ILLIQ = |R| / (|PRC| * VOL); NaN on no-trade days (VOL=0 or approx 0)
# so they are excluded from the monthly mean but still contribute to the return compounding
crsp['illiq_daily'] = np.where(
    crsp['VOL'] > 0,
    crsp['RET'].abs() / (crsp['PRC'].abs() * crsp['VOL']),
    np.nan
)
def calculate_monthly_ret(series):
    return (series + 1).prod() - 1
df_monthly = crsp.groupby(['TICKER', 'Year', 'Month']).agg({
    'illiq_daily': 'mean',
    'RET': calculate_monthly_ret,    
    'VOL': 'sum'                     
}).reset_index()
df_monthly.rename(columns={'RET': 'monthly_ret', 'illiq_daily': 'illiq_avg'}, inplace=True)

# Pull VIX data
today = pd.to_datetime('today').strftime('%Y-%m-%d')
vix = yf.download('^VIX', start='1990-01-01', end=today, interval='1d', progress=False)
if isinstance(vix.columns, pd.MultiIndex):
    vix.columns = ['_'.join(map(str, c)) for c in vix.columns]
close_col = next((c for c in vix.columns if 'close' in str(c).lower()), vix.select_dtypes(include='number').columns[0])
vix['Year'] = pd.to_datetime(vix.index).year
vix['Month'] = pd.to_datetime(vix.index).month
vix_monthly = vix.groupby(['Year','Month'], as_index=False)[close_col].mean().rename(columns={close_col: 'VIX'})
vix_monthly

# ------------ Merging and Final Clean up------------ 
df = df_monthly.merge(hmi[['Year', 'Month', 'HMI_lag1']], on=['Year', 'Month'], how='left')
df = df.merge(treasury, on=['Year', 'Month'], how='left')
df = df.merge(vix_monthly, on=['Year', 'Month'], how='left')
df = df.merge(compustat, left_on=['TICKER', 'Year'], right_on=['tic', 'Year'], how='left').drop(columns=['tic'])
df['log_illiq'] = np.log(df['illiq_avg'])
df = df.sort_values(['TICKER', 'Year', 'Month']).reset_index(drop=True)
treatment_tickers = ['ESS', 'AVB', 'INVH', 'UDR', 'MAA', 'CPT', 'AMH']
control_tickers = ['WELL', 'AMT', 'EQIX', 'PLD']
df['treatment'] = df['TICKER'].apply(lambda x: 1 if x in treatment_tickers else (0 if x in control_tickers else np.nan))
df.columns = df.columns.str.lower()
# df.to_csv('processed_data.csv', index=False)
# mb some of the stocks had a ticker change/ m&a? AMB, HCN etc. weren't part of the query // df[df['treatment'].isna()]['ticker'].unique()

c:\Users\dvolosh\AppData\Local\Programs\Python\Python313\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [47]:
snapshopt = df[df['ticker'] == 'WELL']
snapshopt.head(3).reset_index(drop=True)

,ticker,year,month,illiq_avg,monthly_ret,vol,hmi_lag1,dgs10,vix,debt_to_assets,asset_growth,log_assets,log_illiq,treatment
0,WELL,1993,8,1.509641e-08,0.089110,1389052.0,57,5.677727,11.928636,0.353977,0.260014,5.652573,-18.008809,0.0
1,WELL,1993,9,3.260923e-08,0.199999,1458507.0,60,5.360000,12.931428,0.353977,0.260014,5.652573,-17.238671,0.0
2,WELL,1993,10,8.695833e-08,-0.106061,858088.0,62,5.334000,11.875238,0.353977,0.260014,5.652573,-16.257837,0.0


In [48]:
df[df['ticker'] == 'HCN']

,ticker,year,month,illiq_avg,monthly_ret,vol,hmi_lag1,dgs10,vix,debt_to_assets,asset_growth,log_assets,log_illiq,treatment
2716,HCN,1990,1,2.334528e-07,0.021111,118100.0,43,8.206667,23.347273,NaN,NaN,NaN,-15.270286,NaN
2717,HCN,1990,2,4.238838e-07,-0.017241,49100.0,42,8.473158,23.262632,NaN,NaN,NaN,-14.673807,NaN
2718,HCN,1990,3,2.172327e-07,-0.017544,89100.0,44,8.588636,20.062273,NaN,NaN,NaN,-15.342297,NaN
2719,HCN,1990,4,3.098703e-07,-0.004714,111200.0,40,8.785500,21.403500,NaN,NaN,NaN,-14.987112,NaN
2720,HCN,1990,5,2.463961e-07,-0.055558,185500.0,39,8.758182,18.097727,NaN,NaN,NaN,-15.216326,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3049,HCN,2017,10,4.225230e-11,-0.047239,37669550.0,64,2.360000,10.125455,NaN,NaN,NaN,-23.887362,NaN
3050,HCN,2017,11,3.962787e-11,0.020360,35867970.0,68,2.353333,10.540476,NaN,NaN,NaN,-23.951489,NaN
3051,HCN,2017,12,4.378504e-11,-0.054699,44931432.0,69,2.402500,10.264500,NaN,NaN,NaN,-23.851729,NaN
3052,HCN,2018,1,6.488771e-11,-0.059590,55088723.0,74,2.583810,11.062381,NaN,NaN,NaN,-23.458363,NaN


In [ ]:
df ####

,ticker,year,month,illiq_avg,monthly_ret,vol,hmi_lag1,dgs10,vix,debt_to_assets,asset_growth,log_assets,log_illiq,treatment
0,AMB,1997,11,3.198974e-09,-9.008598e-07,1192300.0,59,5.875000,32.034737,NaN,NaN,NaN,-19.560436,NaN
1,AMB,1997,12,4.131788e-09,9.858649e-02,2484500.0,58,5.808636,26.276364,NaN,NaN,NaN,-19.304555,NaN
2,AMB,1998,1,7.423387e-09,-2.487665e-02,1275200.0,60,5.544500,23.866500,NaN,NaN,NaN,-18.718630,NaN
3,AMB,1998,2,1.068346e-08,-4.081663e-02,933900.0,60,5.574737,19.998947,NaN,NaN,NaN,-18.354569,NaN
4,AMB,1998,3,9.484757e-09,4.151892e-02,1594600.0,65,5.647273,20.158636,NaN,NaN,NaN,-18.473580,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5548,WELL,2024,10,3.520085e-11,5.350338e-02,51547361.0,41,4.095455,19.960870,0.361876,0.000000,10.840449,-24.069951,0.0
5549,WELL,2024,11,2.710933e-11,2.955554e-02,54004044.0,43,4.355789,16.121000,0.361876,0.159777,10.840449,-24.331143,0.0
5550,WELL,2024,11,2.710933e-11,2.955554e-02,54004044.0,43,4.355789,16.121000,0.361876,0.000000,10.840449,-24.331143,0.0
5551,WELL,2024,12,2.523323e-11,-8.792772e-02,62210766.0,46,4.391429,15.866191,0.361876,0.159777,10.840449,-24.402859,0.0


In [ ]:
# Ticker disrepancies and M&A -- why there are tickers we didn't ask for in the data
# ProLogis (NYSE: PLD) acquired AMB Property Corporation (NYSE: AMB) in 2011
# AmerUs Life Holdings Inc (NYSE: AMRS) by Aviva PLC's in 2006
# Welltower Inc. changed its NYSE ticker symbol from HCN to WELL effective at the opening of trading on February 28, 2018

In [6]:
# The earliest date for all tickers with non-na treatment status
df[df['treatment'].notna()].groupby('ticker')['year'].min()
# Seems to make sense to drop INVNH since data available only from 2017; mb EQIX too

ticker
AMH     1990
AMT     1990
CPT     1990
EQIX    2000
ESS     1994
INVH    2017
MAA     1994
PLD     1998
UDR     1990
WELL    1993
Name: year, dtype: int32